# Aint Lacking this time BIATCH  ~~ 

This notebook converts all the physics images of original BUSI dataset into style transferred dataset so that we can train segformer on that physics style transferred dataset and eventually get inference to tackle the domain shift.  

In [1]:
import os
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt 
import json
import streamlit as st
import cv2
import shutil


In [2]:
# First we need to prepare the datasets
import os

from PIL import Image


def get_file_paths(folder):
    image_file_paths = []
    for root, dirs, filenames in os.walk(folder):
        filenames = sorted(filenames)
        for filename in filenames:
            input_path = os.path.abspath(root)
            file_path = os.path.join(input_path, filename)
            if filename.endswith('.png') or filename.endswith('.jpg'):
                image_file_paths.append(file_path)

        break  # prevent descending into subfolders
    return image_file_paths


def align_images(a_file_paths, b_file_paths, target_path):
    if not os.path.exists(target_path):
        os.makedirs(target_path)
    for i in range(len(a_file_paths)):
        img_a = Image.open(a_file_paths[i])
        img_b = Image.open(b_file_paths[i])
        assert(img_a.size == img_b.size)
        # print("file_path_a: ", a_file_paths[i])
        # print("file_path_b: ", b_file_paths[i])
        aligned_image = Image.new("RGB", (img_a.size[0] * 2, img_a.size[1]))
        aligned_image.paste(img_a, (0, 0))
        aligned_image.paste(img_b, (img_a.size[0], 0))
        filename = a_file_paths[i].split("/")[-1].replace(".png","")
        aligned_image.save(os.path.join(target_path, '{}.jpg'.format(filename)))
        # break

In this cell, we place the images inside the test folder to be able to get inference as pix2pix dataset needs to be in some specific shape and that is still shitty way for inference but who cares as if we were deploying this shit to NASA
Less Go

In [ ]:
import os
import shutil

# old dataset folder
# Make sure that this folder has the folder trainA, trainB,
# testA, testB so that we create test folder containing all the images as done below:
dataset_folder = "/home/user/data/phyusformer_data/post_miccai_exps/data/pix2pix_physics_data_inference_all_mixed"
# new dataset folder
# Make sure that this folder has the folder trainA, trainB,
# testA, testB so that we create test folder containing all the images as done below:
dataset_folder = "/home/user/data/phyusformer_data/post_miccai_exps/data/physics_data_high_res/test_all_mixed"
print(dataset_folder)
test_a_path = os.path.join(dataset_folder, "testA")
test_b_path = os.path.join(dataset_folder, "testB")
test_a_file_paths = get_file_paths(test_a_path)
test_b_file_paths = get_file_paths(test_b_path)
assert len(test_a_file_paths) == len(test_b_file_paths)
test_path = os.path.join(dataset_folder, "test")

train_a_path = os.path.join(dataset_folder, "trainA")
train_b_path = os.path.join(dataset_folder, "trainB")
train_a_file_paths = get_file_paths(train_a_path)
train_b_file_paths = get_file_paths(train_b_path)
assert len(train_a_file_paths) == len(train_b_file_paths)
# train_path = os.path.join(dataset_folder, 'train')

align_images(test_a_file_paths, test_b_file_paths, test_path)
align_images(train_a_file_paths, train_b_file_paths, test_path)

/home/user/data/phyusformer_data/post_miccai_exps/data/physics_data_high_res/test_all_mixed


In [7]:
len(os.listdir("/home/user/data/phyusformer_data/post_miccai_exps/data/physics_data_high_res/test_all_mixed/test"))

647

Now go to CLI and run the following command to get the inference, save all the inference images in proper folder:
***
OLD COMMAND 

python test.py --dataroot /home/user/data/phyusformer_data/post_miccai_exps/data/pix2pix_physics_data_inference_all_mixed/ --name pix2pix_source2target --model pix2pix --use_wandb --input_nc 1 --output_nc 1 --wandb_project_name pix2pix_source2target_lowres --gpu_id 0 --checkpoints_dir /home/user/data/phyusformer_data/post_miccai_exps/pix2pix_checkpoints/ --direction AtoB
***

## Copying the Original Masks from the simualted data and placing in the results section 

In [13]:
import os
import shutil
import pickle
import pandas as pd
from PIL import Image
results_dir = "/home/user/haris/pytorch-CycleGAN-and-pix2pix/results/pix2pix_source2target_highres/pix2pix_source2target_highres/test_latest/images"
save_train_split_path = "/home/user/data/phyusformer_data/post_miccai_exps/data/BUSI/processed_data_python_pipeline/train_df.csv"
save_test_split_path = "/home/user/data/phyusformer_data/post_miccai_exps/data/BUSI/processed_data_python_pipeline/test_df.csv"

df_train = pd.read_csv(save_train_split_path)
df_test = pd.read_csv(save_test_split_path)

df_combined = pd.concat([df_train,df_test])
# df_combined.head(10)
for idx,row in df_combined.iterrows():
    mask_path = row['mask_path']
    mask_data = pickle.load(open(mask_path,"rb"))
    mask = mask_data['clean_phantom_binary']
    filename = row['original_filename']
    mask = mask.astype(np.uint8)
    mask = mask * 255
    # save the mask as a png file
    # Image.fromarray(mask).save(os.path.join(results_dir,f"{filename}_mask.png"))
    cv2.imwrite(os.path.join(results_dir,f"{filename}_mask.png"),mask)
    # break

In [15]:
sorted(os.listdir(results_dir))

['benign (1)_fake_B.png',
 'benign (1)_mask.png',
 'benign (1)_real_A.png',
 'benign (1)_real_B.png',
 'benign (10)_fake_B.png',
 'benign (10)_mask.png',
 'benign (10)_real_A.png',
 'benign (10)_real_B.png',
 'benign (100)_fake_B.png',
 'benign (100)_mask.png',
 'benign (100)_real_A.png',
 'benign (100)_real_B.png',
 'benign (101)_fake_B.png',
 'benign (101)_mask.png',
 'benign (101)_real_A.png',
 'benign (101)_real_B.png',
 'benign (102)_fake_B.png',
 'benign (102)_mask.png',
 'benign (102)_real_A.png',
 'benign (102)_real_B.png',
 'benign (103)_fake_B.png',
 'benign (103)_mask.png',
 'benign (103)_real_A.png',
 'benign (103)_real_B.png',
 'benign (104)_fake_B.png',
 'benign (104)_mask.png',
 'benign (104)_real_A.png',
 'benign (104)_real_B.png',
 'benign (105)_fake_B.png',
 'benign (105)_mask.png',
 'benign (105)_real_A.png',
 'benign (105)_real_B.png',
 'benign (106)_fake_B.png',
 'benign (106)_mask.png',
 'benign (106)_real_A.png',
 'benign (106)_real_B.png',
 'benign (107)_fake_B.

Then you can run the following command inside the results folder where the index.html file is generated 
"""
python -m http.server 8000
"""
to visualize the side by side comparison of the generated images 


Once done with it, lets download the data and further process it for 2nd stage that is the training of segformer
Over to you Dahl~